# Hands-On 5: Linear and tree regression

Record a prediction before running each experiment.

In [ ]:
from pathlib import Path
import os
import sys
try:
    import mlcourse.setup
except ModuleNotFoundError:
    bases = [Path(os.environ.get("MLCOURSE_ROOT", Path.cwd())), Path.cwd(), Path("/content/pp-machine-learning")]
    for base in bases:
        for candidate in (base.resolve(), *base.resolve().parents):
            if (candidate / "src/mlcourse/setup.py").is_file():
                sys.path.insert(0, str(candidate / "src"))
                break
        else:
            continue
        break
    else:
        raise RuntimeError("Course files not found. Open the extracted course repository or set MLCOURSE_ROOT to its location.") from None

In [ ]:
from mlcourse.setup import setup_notebook
REPO_ROOT = setup_notebook()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mlcourse.labs import load_course_data

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from mlcourse.labs import regression_metrics
from mlcourse.widgets import interactive_regression

## 1. Inspect a regression dataset

Load the Concrete dataset. Inspect `head()` and the ranges of `cement`, `age` and `strength_mpa`. Strength is measured in MPa.

**Prediction:** What changes when the target is continuous rather than a class label?

*Your response.*

In [ ]:
data = load_course_data('concrete')
display(data.head())
display(data[['cement', 'age', 'strength_mpa']].describe())

**Observation:** Identify the predictors and continuous target.

*Your response.*

**Explanation:** Why is classification accuracy unsuitable here?

*Your response.*

## 2. Create a holdout split

Use `cement` and `age` as predictors. Split once and reuse these observations throughout the comparison.

**Prediction:** Why should every model use the same split?

*Your response.*

In [ ]:
X = data[['cement', 'age']]
y = data.strength_mpa
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, random_state=0)
print(f'Training rows: {len(X_train)}; test rows: {len(X_test)}')

**Observation:** Record the sizes of both partitions.

*Your response.*

**Explanation:** How does reusing a split help compare model behaviour?

*Your response.*

## 3. Fit a line

Fit strength from `cement` alone. Plot the line and test observations.

**Prediction:** Can one line bend to fit different local trends?

*Your response.*

In [ ]:
line = LinearRegression().fit(X_train[['cement']], y_train)
line_predictions = line.predict(X_test[['cement']])
ordered = X_test[['cement']].sort_values('cement')
fig, ax = plt.subplots(figsize=(8, 5), layout='constrained')
ax.scatter(X_test.cement, y_test, s=20, alpha=.6, label='Test observations')
ax.plot(ordered.cement, line.predict(ordered), color='#D55E00', label='Fitted line')
ax.set(xlabel='Cement (kg/m³)', ylabel='Strength (MPa)', title='One-feature linear regression')
ax.legend()
plt.show()
print(f'Slope: {line.coef_[0]:.3f}; intercept: {line.intercept_:.3f}')

**Observation:** Describe the spread around the fitted line.

*Your response.*

**Explanation:** Explain the restriction imposed by a global linear model.

*Your response.*

## 4. Measure regression error

Compute MSE and MAE for the line and a training-mean baseline.

**Prediction:** Which metric gives large errors more influence?

*Your response.*

In [ ]:
baseline = DummyRegressor(strategy='mean').fit(X_train, y_train)
error_table = pd.DataFrame({
    'Training mean': regression_metrics(y_test, baseline.predict(X_test)),
    'Linear model': regression_metrics(y_test, line_predictions),
}).T
display(error_table)

**Observation:** Compare each metric with its baseline value.

*Your response.*

**Explanation:** Explain why MSE and MAE have different numerical scales.

*Your response.*

## 5. Inspect residuals

Plot residuals against predictions, with a zero reference line.

**Prediction:** What would a systematic pattern in the residuals suggest?

*Your response.*

In [ ]:
residuals = y_test - line_predictions
fig, ax = plt.subplots(figsize=(8, 4.5), layout='constrained')
ax.scatter(line_predictions, residuals, alpha=.6, s=22)
ax.axhline(0, color='black', linewidth=1)
ax.set(xlabel='Predicted strength (MPa)', ylabel='Observed minus predicted (MPa)', title='Test residuals')
plt.show()

**Observation:** Describe the spread and any visible pattern around zero.

*Your response.*

**Explanation:** What can residuals reveal that one average error conceals?

*Your response.*

## 6. Compare a line, a plane and a tree

Add `age` to the linear model and compare it with a depth-three regression tree.

**Prediction:** How does a linear plane differ from a tree surface?

*Your response.*

In [ ]:
plane = LinearRegression().fit(X_train, y_train)
small_tree = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X_train, y_train)
for name, model in [('Linear plane', plane), ('Depth-three tree', small_tree)]:
    error_table.loc[name] = regression_metrics(y_test, model.predict(X_test))
display(error_table)

**Observation:** Compare the errors after adding a feature and changing the model family.

*Your response.*

**Explanation:** Distinguish adding information from changing inductive bias.

*Your response.*

## 7. Manipulate the fitted surfaces

Before moving `max_depth`, predict how the tree surface will change. Compare it with the linear surface.

**Prediction:** What happens to the number and size of constant regions as depth increases?

*Your response.*

In [ ]:
regression_lab = interactive_regression(X_train, X_test, y_train, y_test)
display(regression_lab.widget)

**Observation:** Record a setting with visibly finer partitions and its MSE.

*Your response.*

**Explanation:** Explain the surface difference through the models' inductive biases.

*Your response.*

## 8. Make a bounded model comparison

Compare all-feature linear regression with depth-three and depth-ten trees on the same row split.

**Prediction:** Will the configuration with lowest training MSE also have lowest test MSE?

*Your response.*

In [ ]:
all_train = data.drop(columns='strength_mpa').loc[X_train.index]
all_test = data.drop(columns='strength_mpa').loc[X_test.index]
rows = []
for name, model in {'Linear': LinearRegression(),
    'Tree depth 3': DecisionTreeRegressor(max_depth=3, random_state=0),
    'Tree depth 10': DecisionTreeRegressor(max_depth=10, random_state=0)}.items():
    model.fit(all_train, y_train)
    rows.append({'model': name, 'train_MSE': regression_metrics(y_train, model.predict(all_train))['MSE'],
                 **{f'test_{key}': value for key, value in regression_metrics(y_test, model.predict(all_test)).items()}})
comparison = pd.DataFrame(rows).set_index('model')
display(comparison)

**Observation:** Identify the largest train-test gap.

*Your response.*

**Explanation:** Relate that gap to capacity and sensitivity to training observations.

*Your response.*